# 03 - Copy Model to S3

**Elyra Pipeline Node 3** — Copy the trained model and artifacts to an AWS S3 bucket.

**Required** (from pipeline parameters, env, or RHOAI secrets):
- `AWS_ACCESS_KEY_ID`: AWS access key
- `AWS_SECRET_ACCESS_KEY`: AWS secret access key
- `S3_BUCKET`: Target S3 bucket name
- `S3_PREFIX`: Optional object key prefix (default: `models/isolation-forest/`)

**Inputs**:
- `MODEL_PATH`: Path to model pickle
- `FEATURE_COLS_PATH`, `THRESHOLD_PATH`, `TEST_REPORT_PATH`: Optional artifacts to upload

In [ ]:
import os
from pathlib import Path

# AWS credentials - use RHOAI Pipeline Parameters or Kubernetes secrets
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "")
S3_BUCKET = os.getenv("S3_BUCKET", "")
S3_PREFIX = os.getenv("S3_PREFIX", "models/isolation-forest/")

MODEL_PATH = os.getenv("MODEL_PATH", "artifacts/model.pkl")
FEATURE_COLS_PATH = os.getenv("FEATURE_COLS_PATH", "artifacts/feature_cols.json")
THRESHOLD_PATH = os.getenv("THRESHOLD_PATH", "artifacts/threshold.json")
TEST_REPORT_PATH = os.getenv("TEST_REPORT_PATH", "artifacts/test_report.json")

In [ ]:
# Validate credentials
if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise ValueError(
        "AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY must be set. "
        "Configure them as RHOAI pipeline parameters or environment variables."
    )
if not S3_BUCKET:
    raise ValueError("S3_BUCKET must be set.")

print(f"Target bucket: s3://{S3_BUCKET}/{S3_PREFIX}")

In [ ]:
try:
    import boto3
    from botocore.exceptions import ClientError
except ImportError:
    raise ImportError("boto3 required. Install with: pip install boto3")

# Create S3 client
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

print("S3 client initialized")

In [ ]:
def upload_file(local_path: str, s3_key: str) -> str:
    """Upload a file to S3."""
    if not Path(local_path).exists():
        print(f"  [skip] {local_path} not found")
        return ""
    key = f"{S3_PREFIX.rstrip('/')}/{s3_key}"
    s3.upload_file(local_path, S3_BUCKET, key)
    url = f"s3://{S3_BUCKET}/{key}"
    print(f"  Uploaded {local_path} → {url}")
    return url

In [ ]:
# Upload model and artifacts
uploads = []
s3_model_uri = f"s3://{S3_BUCKET}/{S3_PREFIX.rstrip('/')}/"

files_to_upload = [
    (MODEL_PATH, "model.pkl"),
    (FEATURE_COLS_PATH, "feature_cols.json"),
    (THRESHOLD_PATH, "threshold.json"),
    (TEST_REPORT_PATH, "test_report.json"),
]

for local_path, s3_name in files_to_upload:
    url = upload_file(local_path, s3_name)
    if url:
        uploads.append({"local": local_path, "s3": url})

# Export for downstream node (04_deploy_to_serving)
os.environ["S3_MODEL_URI"] = s3_model_uri

In [ ]:
# Summary
print(f"\nCopy complete. {len(uploads)} file(s) uploaded to {s3_model_uri}")
for u in uploads:
    print(f"  - {u['s3']}")
print(f"\nS3_MODEL_URI for deploy node: {s3_model_uri}")